# Historical course notebook
Preserved for provenance. Known split/target issues are documented in ../../docs/LEGACY-REVIEW.md. Outputs are cleared; original scores are not a validated benchmark.


In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np

import xgboost as xgb

btc_market = pd.read_csv('btc-pulse-data.csv') # source: https://www.kaggle.com/datasets/wlwwwlw/bitcoin-pulse-market-trends-and-fear-dataset 

In [ ]:
btc_market['timestamp'] = pd.to_datetime(btc_market['timestamp'])

btc_market["Day"] = btc_market["timestamp"].dt.day.astype(str) 
btc_market["Month"] = btc_market["timestamp"].dt.month.astype(str)
btc_market["Year"] = btc_market["timestamp"].dt.year.astype(str)
btc_market["Time-in-date"] = btc_market["Year"] + "-" + btc_market["Month"] + "-" + btc_market["Day"]

btc_market.head()

In [ ]:
btc_data = btc_market[["timestamp", "Time-in-date", "BTC_USDT_1h_open", "Open_nasdaq", "Open_russell_2000", "Open_vix", "Open_dow_jones", "Open_sp500", "fear_greed_index", "google_trends_bitcoin", "google_trends_buy_crypto", "open_interest", "btc_dominance", "altcoin_market_cap"]]
btc_data = btc_data.drop_duplicates(subset=["Time-in-date"])
btc_data.head()


In [ ]:
btc_data = btc_data.set_index("Time-in-date")
btc_data.head()

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with a larger size
plt.figure(figsize=(12, 6))

# Plot the data with a legend
btc_data[['BTC_USDT_1h_open', 'Open_nasdaq', 'Open_russell_2000', 'Open_vix', 'Open_dow_jones', 'Open_sp500']].plot()

# Add labels and title
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Market Indicators Over Time')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
btc_data[['BTC_USDT_1h_open', 'Open_nasdaq', 'Open_russell_2000', 'Open_vix', 'Open_dow_jones', 'Open_sp500', 'fear_greed_index', 'google_trends_bitcoin', 'google_trends_buy_crypto', 'open_interest', 'btc_dominance', 'altcoin_market_cap']].corr()

I'll help create a function to generate lagged features for our Bitcoin prediction model. Given the data we have, we'll create a function that can generate lagged values for our features.

In [ ]:
def lagged_features(df, features, lag_periods=[1, 2]):

    df_copy = df.copy()
    
    for feature in features:
        for lag in lag_periods:
            lagged_name = f'{feature}_lag_{lag}'
            df_copy[lagged_name] = df_copy[feature].shift(lag)
    df_copy = df_copy.dropna()
    return df_copy


features_to_lag = ['BTC_USDT_1h_open', 'Open_nasdaq', 'Open_russell_2000', 
                  'Open_vix', 'Open_dow_jones', 'Open_sp500', 
                  'fear_greed_index', 'google_trends_bitcoin', 
                  'google_trends_buy_crypto', 'open_interest', 
                  'btc_dominance', 'altcoin_market_cap']

# Create lagged features with 1-day and 3-day lags
btc_data_lagged = lagged_features(btc_data, features_to_lag)



testing_data = btc_data[btc_data.timestamp.dt.year == 2025]
training_data = btc_data[btc_data.timestamp.dt.year < 2025]
train_data = lagged_features(training_data, features_to_lag, lag_periods=[1, 2])
test_data = lagged_features(testing_data, features_to_lag, lag_periods=[1, 2])

In [ ]:
train_data.head()

In [ ]:
useful_cols = [x for x in train_data if x not in ['Open_vix_lag_1', 'Open_vix_lag_2', 'fear_greed_index_lag_1', 'fear_greed_index_lag_2', 'google_trends_bitcoin_lag_1', 'google_trends_bitcoin_lag_2', 'google_trends_buy_crypto_lag_1', 'google_trends_buy_crypto_lag_2', 'btc_dominance_lag_1', 'btc_dominance_lag_2']]

In [ ]:
train_data = train_data[useful_cols]
test_data = test_data[useful_cols]

In [ ]:
train_data.head()

I've created a function `lagged_features` that generates lagged versions of specified features. The function:
1. Takes a DataFrame, list of features, and lag periods as input
2. Creates new columns with lagged values for each feature and lag period
3. Handles data cleaning by removing NaN values created by the lag operation
4. Returns the processed DataFrame

In the example usage, I applied 1-day and 2-day lags to all our numerical features. The resulting dataset now has 20 additional columns (10 features × 2 lag periods), which can be useful for time series analysis and prediction tasks.

In [ ]:
test = test_data.drop(['Open_nasdaq', 'Open_russell_2000', 'Open_vix', 'Open_dow_jones', 'Open_sp500', 'fear_greed_index', 'google_trends_bitcoin', 'google_trends_buy_crypto', 'open_interest', 'btc_dominance', 'altcoin_market_cap'], axis=1)
train = train_data.drop(['Open_nasdaq', 'Open_russell_2000', 'Open_vix', 'Open_dow_jones', 'Open_sp500', 'fear_greed_index', 'google_trends_bitcoin', 'google_trends_buy_crypto', 'open_interest', 'btc_dominance', 'altcoin_market_cap'], axis=1)

In [ ]:
train.head()

In [ ]:
train.corr()

In [ ]:

# Variables to use:
# NASAQ, Russell 2000, Dow Jones, SP500, open interest

btc_data_used = [x for x in btc_data_lagged.columns if x not in ['Open_vix_lag_1', 'Open_vix_lag_2', 'fear_greed_index_lag_1', 'fear_greed_index_lag_2', 'google_trends_bitcoin_lag_1', 'google_trends_bitcoin_lag_2', 'google_trends_buy_crypto_lag_1', 'google_trends_buy_crypto_lag_2', 'btc_dominance_lag_1', 'btc_dominance_lag_2']]

In [ ]:
btc_data_final = btc_data_lagged[btc_data_used]

In [ ]:

btc_data_final.index = btc_data_final['timestamp']
btc_data_final.drop('timestamp', axis=1, inplace=True)
btc_data_final.head()

In [ ]:
btc_data_final["year"] = btc_data_final.index.year
btc_data_final.head()

testing_data = btc_data_final[btc_data_final.year == 2025]
training_data = btc_data_final[btc_data_final.year < 2025]

In [ ]:
x_col = [x for x in train.columns if x not in ["BTC_USDT_1h_open", "year", "timestamp"]]

y_col = ["BTC_USDT_1h_open"]


In [ ]:
x_col

In [ ]:
X_train = train[x_col]
X_test = test[x_col]

y_train = train[y_col]
y_test = test[y_col]

In [ ]:
X_train.head()

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=250, max_depth=5, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

In [ ]:
plot = pd.DataFrame({'Actual': y_test['BTC_USDT_1h_open'], 'Predicted': y_pred})
plot.plot(figsize=(10,5), xlabel='Date', ylabel='Price', title='Actual vs. Predicted Bitcoin Price')

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
np.sqrt(mean_squared_error(y_test, y_pred))